# 260401 검색 평가 메트릭 (Precision, Recall, F1, BLEU) & LLM-as-Judge

**4주차 Day 2** | RAG 시스템의 검색 품질과 생성 품질을 정량적으로 평가하는 방법을 배운다.

**오늘 다루는 내용:**
- Precision@K, Recall@K, F1@K -- 검색이 얼마나 정확하고 빠짐없는지
- R-Precision -- 쿼리마다 공정하게 비교하는 Precision
- F-beta -- 상황에 따라 Precision/Recall 가중치 조절
- BLEU Score -- 생성 답변의 n-gram 기반 정밀도 (번역 평가에서 유래)
- LLM-as-Judge (Context Relevance) -- LLM이 직접 검색 결과의 관련성을 채점

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w4_rag_evaluation/llm_260401_retrieval_metrics.ipynb)

---
## 0. 환경 설정

In [ ]:
# Colab 사용 시 아래 주석 해제
# !pip install -q openai langchain langchain-openai langchain-community faiss-cpu python-dotenv numpy pandas matplotlib

# import os
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple, Any
from collections import Counter
from dotenv import load_dotenv

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

load_dotenv()

LLM_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
llm = ChatOpenAI(model=LLM_MODEL)
embeddings_model = OpenAIEmbeddings(model=EMBEDDING_MODEL)

---
## 1. 평가용 문서 & QA 데이터셋 준비

검색 평가를 하려면 두 가지가 필요하다:
1. **문서 모음** (검색 대상) -- 도서관의 책들
2. **QA 데이터셋** (질문 + 정답 문서 ID) -- 시험지의 정답지

정답 문서를 미리 알아야 "우리 검색 엔진이 정답을 잘 찾았는지" 채점할 수 있다.

In [ ]:
# 10개의 LLM/AI 관련 문서 -- 각각 고유 doc_id를 가진다
documents = [
    {"doc_id": "doc_001", "title": "트랜스포머 아키텍처",
     "content": "트랜스포머는 2017년 'Attention is All You Need' 논문에서 제안된 아키텍처입니다. "
                "셀프 어텐션 메커니즘을 사용하여 입력 시퀀스의 모든 위치 간 관계를 병렬로 처리합니다. "
                "인코더-디코더 구조로 구성되며, 멀티헤드 어텐션과 피드포워드 네트워크가 핵심 구성요소입니다."},
    {"doc_id": "doc_002", "title": "RAG 시스템 개요",
     "content": "RAG(Retrieval-Augmented Generation)는 검색과 생성을 결합한 기법입니다. "
                "외부 지식 베이스에서 관련 문서를 검색한 후, 이를 컨텍스트로 활용하여 LLM이 답변을 생성합니다. "
                "환각(hallucination)을 줄이고 최신 정보를 반영할 수 있는 장점이 있습니다."},
    {"doc_id": "doc_003", "title": "벡터 임베딩과 유사도 검색",
     "content": "벡터 임베딩은 텍스트를 고차원 벡터 공간에 매핑하는 기술입니다. "
                "코사인 유사도를 사용하여 의미적으로 유사한 문서를 검색합니다. "
                "FAISS, Pinecone 등의 벡터 데이터베이스가 대규모 검색에 활용됩니다."},
    {"doc_id": "doc_004", "title": "프롬프트 엔지니어링",
     "content": "프롬프트 엔지니어링은 LLM에게 효과적인 지시를 설계하는 기술입니다. "
                "Few-shot, Chain-of-Thought, Zero-shot 등의 기법이 있습니다. "
                "시스템 프롬프트와 사용자 프롬프트를 구분하여 역할과 지시를 명확히 합니다."},
    {"doc_id": "doc_005", "title": "파인튜닝과 전이학습",
     "content": "파인튜닝은 사전 학습된 모델을 특정 태스크에 맞게 추가 학습시키는 기법입니다. "
                "LoRA, QLoRA 등의 효율적 파인튜닝 기법이 대형 모델 적응에 널리 사용됩니다. "
                "전이학습을 통해 적은 데이터로도 높은 성능을 달성할 수 있습니다."},
    {"doc_id": "doc_006", "title": "토큰화와 텍스트 전처리",
     "content": "토큰화는 텍스트를 모델이 처리할 수 있는 단위로 분할하는 과정입니다. "
                "BPE(Byte Pair Encoding), WordPiece, SentencePiece 등의 알고리즘이 있습니다. "
                "한국어는 교착어 특성상 형태소 분석 기반 토큰화가 효과적입니다."},
    {"doc_id": "doc_007", "title": "LLM 평가 메트릭",
     "content": "LLM 평가에는 자동 메트릭과 인간 평가가 사용됩니다. "
                "BLEU, ROUGE, BERTScore 등의 자동 메트릭은 참조 답변과의 유사도를 측정합니다. "
                "LLM-as-Judge 방식은 다른 LLM을 활용하여 품질을 평가하는 최신 접근법입니다."},
    {"doc_id": "doc_008", "title": "청킹 전략",
     "content": "청킹은 긴 문서를 적절한 크기로 분할하는 전략입니다. "
                "고정 크기 청킹, 의미 기반 청킹, 재귀적 청킹 등의 방법이 있습니다. "
                "청크 크기와 오버랩은 검색 품질에 큰 영향을 미치며, 보통 500-1000 토큰을 사용합니다."},
    {"doc_id": "doc_009", "title": "하이브리드 검색",
     "content": "하이브리드 검색은 키워드 검색(BM25)과 벡터 검색을 결합한 방식입니다. "
                "RRF(Reciprocal Rank Fusion)를 통해 두 검색 결과를 효과적으로 병합합니다. "
                "키워드 매칭의 정확성과 시맨틱 검색의 의미 이해를 동시에 활용합니다."},
    {"doc_id": "doc_010", "title": "멀티모달 AI",
     "content": "멀티모달 AI는 텍스트, 이미지, 오디오 등 다양한 데이터 유형을 처리합니다. "
                "GPT-4V, Gemini 등이 대표적인 멀티모달 모델입니다. "
                "CLIP 모델은 이미지-텍스트 쌍을 학습하여 크로스모달 검색에 활용됩니다."},
]

In [ ]:
# QA 데이터셋: 각 질문에 대해 "정답 문서 ID"가 라벨링되어 있다
# relevant_doc_ids = 이 질문의 정답에 해당하는 문서들
# ground_truth = 기대되는 정답 텍스트 (나중에 BLEU 등에서 사용)
qa_dataset = [
    {"query_id": "q01", "question": "트랜스포머의 핵심 메커니즘은 무엇인가요?",
     "relevant_doc_ids": ["doc_001"],
     "ground_truth": "트랜스포머의 핵심 메커니즘은 셀프 어텐션으로, 입력 시퀀스의 모든 위치 간 관계를 병렬로 처리합니다."},
    {"query_id": "q02", "question": "RAG 시스템의 장점은 무엇인가요?",
     "relevant_doc_ids": ["doc_002"],
     "ground_truth": "RAG는 환각을 줄이고 최신 정보를 반영할 수 있으며, 외부 지식 베이스를 활용하여 더 정확한 답변을 생성합니다."},
    {"query_id": "q03", "question": "벡터 유사도 검색에 사용되는 데이터베이스는?",
     "relevant_doc_ids": ["doc_003"],
     "ground_truth": "FAISS, Pinecone 등의 벡터 데이터베이스가 대규모 벡터 유사도 검색에 활용됩니다."},
    {"query_id": "q04", "question": "프롬프트 엔지니어링의 주요 기법은?",
     "relevant_doc_ids": ["doc_004"],
     "ground_truth": "Few-shot, Chain-of-Thought, Zero-shot 등의 기법이 있으며, 시스템/사용자 프롬프트를 구분합니다."},
    {"query_id": "q05", "question": "효율적 파인튜닝 기법에는 어떤 것이 있나요?",
     "relevant_doc_ids": ["doc_005"],
     "ground_truth": "LoRA, QLoRA 등의 효율적 파인튜닝 기법이 대형 모델 적응에 널리 사용됩니다."},
    {"query_id": "q06", "question": "한국어 토큰화에 적합한 방법은?",
     "relevant_doc_ids": ["doc_006"],
     "ground_truth": "한국어는 교착어 특성상 형태소 분석 기반 토큰화가 효과적입니다."},
    {"query_id": "q07", "question": "LLM-as-Judge란 무엇인가요?",
     "relevant_doc_ids": ["doc_007"],
     "ground_truth": "LLM-as-Judge는 다른 LLM을 활용하여 생성 품질을 평가하는 최신 접근법입니다."},
    {"query_id": "q08", "question": "적절한 청크 크기는 얼마인가요?",
     "relevant_doc_ids": ["doc_008"],
     "ground_truth": "보통 500-1000 토큰 크기를 사용하며, 청크 크기와 오버랩이 검색 품질에 큰 영향을 미칩니다."},
    {"query_id": "q09", "question": "하이브리드 검색에서 결과를 병합하는 방법은?",
     "relevant_doc_ids": ["doc_009"],
     "ground_truth": "RRF(Reciprocal Rank Fusion)를 통해 키워드 검색과 벡터 검색 결과를 효과적으로 병합합니다."},
    {"query_id": "q10", "question": "검색과 생성을 결합하여 환각을 줄이는 기법은?",
     "relevant_doc_ids": ["doc_002", "doc_003"],
     "ground_truth": "RAG(Retrieval-Augmented Generation)는 검색으로 관련 문서를 찾고 LLM이 이를 기반으로 답변을 생성하여 환각을 줄입니다."},
    {"query_id": "q11", "question": "CLIP 모델의 활용 분야는?",
     "relevant_doc_ids": ["doc_010"],
     "ground_truth": "CLIP은 이미지-텍스트 쌍을 학습하여 크로스모달 검색에 활용됩니다."},
    {"query_id": "q12", "question": "트랜스포머의 구조는 어떻게 되어있나요?",
     "relevant_doc_ids": ["doc_001"],
     "ground_truth": "인코더-디코더 구조로 구성되며, 멀티헤드 어텐션과 피드포워드 네트워크가 핵심 구성요소입니다."},
]

---
## 2. 벡터스토어 구축 & 검색 결과 캐싱

문서를 FAISS 벡터스토어에 넣고, 모든 질문에 대해 미리 검색을 돌려둔다.  
메트릭 계산할 때마다 검색을 다시 안 하려고 **캐시(cache)**에 저장해두는 것.

In [ ]:
# 문서 딕셔너리 -> LangChain Document 객체로 변환 후 FAISS에 저장
langchain_docs = [
    Document(page_content=doc["content"], metadata={"doc_id": doc["doc_id"], "title": doc["title"]})
    for doc in documents
]
vectorstore = FAISS.from_documents(langchain_docs, embeddings_model)

# 검색 함수: 쿼리에 대해 상위 k개 문서를 유사도 점수와 함께 반환
def search_documents(query: str, k: int = 5) -> List[Dict]:
    results = vectorstore.similarity_search_with_score(query, k=k)
    retrieved = []
    for doc, score in results:
        retrieved.append({
            "doc_id": doc.metadata["doc_id"],
            "title": doc.metadata["title"],
            "content": doc.page_content,
            "score": float(score)
        })
    return retrieved

# 모든 질문에 대해 상위 5개 검색 결과를 미리 캐싱
# key: query_id, value: 검색된 문서 리스트
search_results_cache = {}
for qa in qa_dataset:
    results = search_documents(qa["question"], k=5)
    search_results_cache[qa["query_id"]] = results

print(f"캐싱 완료: {len(search_results_cache)}개 쿼리")

In [ ]:
# 캐시 내용 확인 -- q01의 검색 결과 예시
search_results_cache["q01"]

---
## 3. Precision@K -- "내가 보여준 것 중 정답 비율"

**비유: 코로나 진단키트**  
양성(Positive)이라고 판정한 200명 중 실제 환자가 100명이면 Precision = 50%  

**검색에 대입하면:**  
K개 검색 결과를 보여줬는데, 그 중 실제 관련 문서가 몇 개?  
- 분자: 관련 문서 & 검색된 문서 (교집합)  
- 분모: K (보여준 개수)  

> K가 커질수록 관련 없는 문서가 섞여 들어와서 Precision은 낮아지는 경향이 있다.  
> (그물을 크게 던지면 쓰레기도 같이 잡힌다)

In [ ]:
def precision_at_k(query_id, k):
    """Precision@K: 상위 K개 검색 결과 중 실제 관련 문서의 비율"""
    # 1. 이 쿼리의 정답 문서 ID를 가져온다
    qa = next(q for q in qa_dataset if q['query_id'] == query_id)
    relevant_ids = set(qa['relevant_doc_ids'])  # 정답 문서 ID 집합

    # 2. 캐시에서 상위 K개 검색 결과를 가져온다
    retrieved = search_results_cache[query_id][:k]
    retrieved_ids = {r['doc_id'] for r in retrieved}  # 검색된 문서 ID 집합

    # 3. 교집합 = 정답이면서 검색도 된 문서
    relevant_retrieved = relevant_ids & retrieved_ids

    # 4. 교집합 크기 / K
    return len(relevant_retrieved) / k

---
## 4. Recall@K -- "진짜 정답 중 내가 찾아낸 비율"

**비유: 코로나 환자 중 검출률**  
실제 환자 100명 중 진단키트가 잡아낸 게 80명이면 Recall = 80%  

**검색에 대입하면:**  
전체 관련 문서 중 K개 검색 결과에 포함된 비율  
- 분자: 관련 문서 & 검색된 문서 (교집합)  
- 분모: 전체 관련 문서 수 (R)  

> K가 커질수록 더 많이 찾아내므로 Recall은 높아진다.  
> (그물을 크게 던지면 물고기를 놓칠 확률이 줄어든다)

In [ ]:
def recall_at_k(query_id, k):
    """Recall@K: 전체 관련 문서 중 상위 K개에 포함된 비율"""
    qa = next(q for q in qa_dataset if q['query_id'] == query_id)
    relevant_ids = set(qa['relevant_doc_ids'])

    retrieved = search_results_cache[query_id][:k]
    retrieved_ids = {r['doc_id'] for r in retrieved}

    relevant_retrieved = relevant_ids & retrieved_ids

    # Precision과 다른 점: 분모가 K가 아니라 "전체 관련 문서 수"
    return len(relevant_retrieved) / len(relevant_ids) if relevant_ids else 0.0

---
## 5. Precision vs Recall -- 트레이드오프 관계

**핵심 직관: 파이 하나를 나눠 먹는 구조**

| | Precision 높이려면 | Recall 높이려면 |
|---|---|---|
| 전략 | 확실한 것만 보여주기 (K 작게) | 최대한 많이 보여주기 (K 크게) |
| 비유 | 큰 물고기만 잡는 촘촘한 그물 | 모든 걸 쓸어담는 넓은 그물 |
| 부작용 | 관련 문서를 놓칠 수 있음 | 노이즈(쓰레기)가 많아짐 |

**RAG에서의 의미:**
- Precision 낮으면 -> 노이즈가 LLM에 들어가 -> hallucination 발생
- Recall 낮으면 -> 핵심 정보 누락 -> 불완전한 답변 생성

---
## 6. F1@K -- Precision과 Recall의 조화평균

둘 중 하나만 높으면 F1은 낮게 나온다. **둘 다 높아야** F1이 높다.  
수식: `F1 = 2 * P * R / (P + R)`  

> 산술평균이 아니라 **조화평균**이라서 한쪽이 0이면 전체가 0이 된다.

In [ ]:
def f1_at_k(query_id, k):
    """F1@K: Precision@K와 Recall@K의 조화평균"""
    p = precision_at_k(query_id, k)
    r = recall_at_k(query_id, k)

    if p + r == 0:
        return 0.0

    # 조화평균: 둘 중 하나가 0이면 전체가 0
    return 2 * p * r / (p + r)

---
## 7. R-Precision -- 쿼리마다 공정한 Precision

**Precision@K의 한계:**  
쿼리마다 관련 문서 수가 다른데 K를 고정하면 불공정하다.  
- 관련 문서 1개인 쿼리에 K=5 -> 아무리 잘해도 Precision = 1/5 = 0.2  
- 관련 문서 10개인 쿼리에 K=5 -> 너무 적게 봄  

**해결: K 대신 R(관련 문서 수)을 사용!**  
관련 문서가 3개면 상위 3개만, 7개면 상위 7개만 평가한다.  
수식상 Recall@R과 값은 같지만, "공정한 Precision"이라는 관점이 다르다.

In [ ]:
def r_precision(query_id):
    """R-Precision: 관련 문서 수 R개만큼만 보고 Precision 계산 (K 불필요)"""
    qa = next(q for q in qa_dataset if q['query_id'] == query_id)
    relevant_ids = set(qa['relevant_doc_ids'])
    R = len(relevant_ids)  # 이 쿼리의 관련 문서 수 = K 대신 사용할 값

    if R == 0:
        return 0.0

    # R개를 K로 사용하여 Precision 계산
    return precision_at_k(query_id, R)

---
## 8. F-beta@K -- 상황에 따라 Precision/Recall 가중치 조절

F1은 P와 R을 동등하게 보지만, 실무에서는 한쪽이 더 중요할 수 있다.

| beta 값 | 의미 | 사용 예시 |
|---|---|---|
| beta < 1 (예: 0.5) | **Precision** 더 중시 | RAG 컨텍스트 검색 (노이즈 방지), 스팸 필터 |
| beta = 1 | P와 R 동등 = F1 | 일반적인 상황 |
| beta > 1 (예: 2.0) | **Recall** 더 중시 | 의료 검색 (하나라도 놓치면 안 됨) |

수식: `F_beta = (1 + beta^2) * P * R / (beta^2 * P + R)`

In [ ]:
def f_beta_at_k(query_id, k, beta):
    """F-beta@K: beta로 Precision/Recall 가중치를 조절하는 메트릭"""
    p = precision_at_k(query_id, k)
    r = recall_at_k(query_id, k)

    beta_sq = beta ** 2  # beta 제곱

    # beta=1이면 분자=(1+1)*P*R=2PR, 분모=(1*P+R)=P+R -> F1과 동일
    if beta_sq * p + r == 0:
        return 0.0
    return (1 + beta_sq) * p * r / (beta_sq * p + r)

---
## 9. 전체 메트릭 계산 -- K별 평균 성능 확인

K=1, 3, 5에서 전체 쿼리의 평균 Precision, Recall, F1을 비교한다.

In [ ]:
# K=1, 3, 5에서 평균 P, R, F1 비교
# K=1: 딱 1개만 보여줄 때 -> Precision 높고, Recall 낮음
# K=5: 5개 보여줄 때 -> Precision 낮고, Recall 높음 (트레이드오프!)
for k in [1, 3, 5]:
    avg_p = np.mean([precision_at_k(qa['query_id'], k) for qa in qa_dataset])
    avg_r = np.mean([recall_at_k(qa['query_id'], k) for qa in qa_dataset])
    avg_f1 = np.mean([f1_at_k(qa['query_id'], k) for qa in qa_dataset])
    print(f"K={k}: P@K={avg_p:.4f}, R@K={avg_r:.4f}, F1@K={avg_f1:.4f}")

In [ ]:
# R-Precision 평균 (K에 관계없이 쿼리마다 공정하게 평가)
avg_rp = np.mean([r_precision(qa['query_id']) for qa in qa_dataset])
print(f"평균 R-Precision: {avg_rp:.4f}")

In [ ]:
# F-beta: beta 값에 따른 변화 관찰 (K=3 고정)
# beta=0.5 -> Precision 중시, beta=2.0 -> Recall 중시
for beta in [0.5, 1.0, 2.0]:
    avg_fb = np.mean([f_beta_at_k(qa['query_id'], 3, beta) for qa in qa_dataset])
    print(f"beta={beta}, F{beta}@3: {avg_fb:.4f}")

---
## 10. 쿼리별 상세 결과 테이블

각 질문마다 개별 성능을 표로 확인한다. 특정 쿼리에서 성능이 낮다면 그 쿼리의 검색 결과를 살펴보면 된다.

In [ ]:
# 쿼리별 P@3, R@3, F1@3, R-Precision을 DataFrame으로 정리
rows = []
for qa in qa_dataset:
    qid = qa['query_id']
    rows.append({
        'query_id': qid,
        'question': qa['question'][:30],
        'P@3': precision_at_k(qid, 3),
        'R@3': recall_at_k(qid, 3),
        'F1@3': f1_at_k(qid, 3),
        'R-Precision': r_precision(qid)
    })

pd.DataFrame(rows)

---
## 11. K=1~5 전체 비교 + F-beta 변화 관찰

K를 1부터 5까지 바꿔가며 평균 Precision, Recall, F-beta(0.5), F-beta(2.0)를 계산한다.  
K가 커질수록 Precision은 떨어지고 Recall은 올라가는 트레이드오프를 눈으로 확인!

In [ ]:
# K=1~5 범위에서 평균 메트릭 계산
k_range = list(range(1, 6))

avg_p = [np.mean([precision_at_k(qa['query_id'], k) for qa in qa_dataset]) for k in k_range]
avg_r = [np.mean([recall_at_k(qa['query_id'], k) for qa in qa_dataset]) for k in k_range]
avg_beta_05 = [np.mean([f_beta_at_k(qa['query_id'], k, 0.5) for qa in qa_dataset]) for k in k_range]
avg_beta_2 = [np.mean([f_beta_at_k(qa['query_id'], k, 2.0) for qa in qa_dataset]) for k in k_range]

# 결과 확인
for i, k in enumerate(k_range):
    print(f"K={k}: P={avg_p[i]:.4f}, R={avg_r[i]:.4f}, F0.5={avg_beta_05[i]:.4f}, F2.0={avg_beta_2[i]:.4f}")

---
## 12. BLEU Score -- 생성 답변의 n-gram 기반 정밀도

여기서부터는 **검색이 아니라 생성**을 평가한다.  
BLEU는 원래 기계번역 품질을 재는 메트릭 (Bilingual Evaluation Understudy).

**핵심 아이디어:**  
생성한 답변의 n-gram(연속 n개 단어)이 정답에 얼마나 등장하는지 본다.

| n-gram | 예시 ("트랜스포머는 어텐션 메커니즘을 활용합니다") |
|---|---|
| 1-gram (unigram) | 트랜스포머는 / 어텐션 / 메커니즘을 / 활용합니다 |
| 2-gram (bigram) | 트랜스포머는 어텐션 / 어텐션 메커니즘을 / 메커니즘을 활용합니다 |
| 3-gram (trigram) | 트랜스포머는 어텐션 메커니즘을 / 어텐션 메커니즘을 활용합니다 |

> n이 커질수록 연속으로 일치해야 하므로 점수가 낮아진다.  
> Brevity Penalty: 너무 짧은 답변으로 치팅하는 것을 방지.

In [ ]:
def get_ngrams(tokens, n):
    """토큰 리스트에서 n-gram을 추출하여 Counter로 반환"""
    # 예: tokens=["나는","학교에","간다"], n=2
    #   -> ("나는","학교에"), ("학교에","간다") 두 개의 bigram
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))

In [ ]:
def bleu_score(reference, hypothesis, max_n):
    """
    BLEU Score 계산
    - reference: 정답 텍스트
    - hypothesis: 생성된 텍스트 (우리 RAG의 답변)
    - max_n: 최대 n-gram 크기 (보통 4)
    """
    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()

    # --- Brevity Penalty (BP) ---
    # 생성문이 정답보다 짧으면 페널티 부과 (짧게 써서 치팅 방지)
    c, r = len(hyp_tokens), len(ref_tokens)  # c=생성 길이, r=정답 길이
    bp = 1.0 if c >= r else np.exp(1 - r/c) if c > 0 else 0.0

    # --- 각 n에 대해 Precision 계산 ---
    precision = {}
    for n in range(1, max_n + 1):
        ref_ngrams = get_ngrams(ref_tokens, n)   # 정답의 n-gram
        hyp_ngrams = get_ngrams(hyp_tokens, n)   # 생성의 n-gram

        if len(hyp_ngrams) == 0:
            precision[n] = 0.0
            continue

        # clipped count: 같은 n-gram을 반복 생성해서 치팅하는 것 방지
        # 생성 n-gram 개수를 정답 n-gram 개수로 잘라줌 (clip)
        clipped = sum(min(cnt, ref_ngrams.get(ng, 0)) for ng, cnt in hyp_ngrams.items())
        precision[n] = clipped / sum(hyp_ngrams.values())

    # --- 기하평균으로 최종 BLEU 계산 ---
    # log를 취해서 더한 뒤 exp -> 기하평균
    log_avg, valid_n = 0.0, 0
    for n in range(1, max_n + 1):
        if precision[n] > 0:
            log_avg += np.log(precision[n])
            valid_n += 1

    bleu = bp * np.exp(log_avg / valid_n) if valid_n > 0 else 0.0

    return {'bleu': round(bleu, 4), 'bp': round(bp, 4)}

In [ ]:
# BLEU Score 테스트
# 정답과 생성문이 비슷하지만 완전히 같지는 않은 예시
ref = "트랜스포머의 핵심 메커니즘은 셀프 어텐션으로, 입력 시퀀스의 모든 위치 간 관계를 병렬로 처리합니다."
hyp = "트랜스포머는 셀프 어텐션 메커니즘을 사용하여 시퀀스 내 모든 위치 간 관계를 병렬로 처리합니다."

result = bleu_score(ref, hyp, 4)
print(f"BLEU: {result['bleu']}, BP: {result['bp']}")
# BP=1.0 -> 생성문이 정답보다 짧지 않아서 페널티 없음

---
## 13. 전통 메트릭의 한계

BLEU, ROUGE 같은 전통 메트릭은 **글자 단위 비교**라서 치명적 한계가 있다:

| 정답 | 생성 | 의미 | BLEU 결과 |
|---|---|---|---|
| 환각 | hallucination | 같은 뜻 | 0점 (글자가 다르니까) |
| 오토바이 | 바이크 | 같은 뜻 | 0점 |

**장점:** 빠르고, 비용 없고, 재현 가능 (같은 입력 = 같은 출력)  
**단점:** 동의어에 약하고, 정답 텍스트가 반드시 필요  

이 한계를 극복하기 위해 **BERTScore** (임베딩 기반 의미 유사도)나 **LLM-as-Judge**를 사용한다.

---
## 14. LLM-as-Judge: Context Relevance 평가

LLM에게 "이 검색 결과가 질문에 얼마나 관련 있어?" 하고 직접 채점시키는 방법.  

**정량 메트릭 vs LLM-as-Judge:**

| | 정량 메트릭 (P, R, BLEU) | LLM-as-Judge |
|---|---|---|
| 속도 | 빠름 | 느림 (API 호출) |
| 비용 | 무료 | LLM 비용 발생 |
| 재현성 | 항상 동일 결과 | 모델/프롬프트에 따라 달라질 수 있음 |
| 의미 이해 | 글자 비교만 가능 | 동의어, 맥락까지 이해 |

> 실무에서는 **둘 다 함께** 사용하는 것이 좋다.

In [ ]:
def evaluate_context_relevance(question, contexts, model):
    """LLM을 사용하여 검색된 컨텍스트의 관련성 평가"""

    # 검색된 문서들을 [컨텍스트 1], [컨텍스트 2], ... 형태로 포매팅
    context_text = '\n\n'.join([
        f"[컨텍스트 {i+1}] : {ctx}" for i, ctx in enumerate(contexts)
    ])

    # 프롬프트: 각 컨텍스트의 관련성을 1-5점으로 평가하도록 지시
    # JSON 형식으로 응답하게 해서 파싱이 쉽도록 함
    prompt = f"""질문과 검색된 컨텍스트의 관련성을 평가하세요.

    ## 질문
    {question}

    ## 검색된 컨텍스트
    {context_text}

    ## 평가 기준
    각 컨텍스트에 대해 1-5점으로 평가하세요:
    - 5 : 질문에 직접적으로 답할 수 있는 핵심 정보 포함
    - 4 : 질문과 매우 관련 있는 정보 포함
    - 3 : 부분적으로 관련 있는 정보 포함
    - 2 : 간접적으로만 관련 있음
    - 1 : 거의 관련 없음

    ## 응답 형식 (JSON)
    {{"scores" : [점수1, 점수2, ...], "average" : 평균점수, "reasoning" : "평가 근거"}}

    반드시 유효한 JSON만 출력하세요."""

    # response_format으로 JSON 출력을 강제
    response = model.bind(response_format={'type': 'json_object'}).invoke([HumanMessage(content=prompt)])
    result = json.loads(response.content)

    return result

In [ ]:
# 샘플 테스트: q01 (트랜스포머의 핵심 메커니즘은?)
sample_qa = qa_dataset[0]
print(f"질문: {sample_qa['question']}")
print(f"정답 문서: {sample_qa['relevant_doc_ids']}")

In [ ]:
# 상위 3개 검색 결과의 content를 추출하여 LLM에게 평가시킴
sample_results = search_results_cache[sample_qa['query_id']][:3]
sample_contexts = [r['content'] for r in sample_results]

# 어떤 문서가 검색되었는지 확인
for i, r in enumerate(sample_results):
    print(f"검색 {i+1}: {r['doc_id']} - {r['title']}")

In [ ]:
# LLM-as-Judge 실행 (API 호출 발생)
cr_result = evaluate_context_relevance(sample_qa['question'], sample_contexts, llm)
cr_result

In [ ]:
# 결과를 보기 좋게 출력
# doc_001(트랜스포머)만 5점, 나머지는 1점으로 나올 것
for i, (r, score) in enumerate(zip(sample_results, cr_result.get('scores', []))):
    print(f"컨텍스트 {i+1} ({r['doc_id']}): {score}/5")

print(f"\n평가 근거: {cr_result.get('reasoning', '')}")

---
## 핵심 정리

| 메트릭 | 한마디 요약 | 분모 |
|---|---|---|
| **Precision@K** | K개 보여줬는데 그 중 맞은 비율 | K |
| **Recall@K** | 전체 정답 중 찾아낸 비율 | R (전체 관련 문서 수) |
| **F1@K** | P와 R의 조화평균 (균형 지표) | - |
| **R-Precision** | R개만 보고 평가 (공정한 P) | R |
| **F-beta** | beta로 P/R 가중치 조절 | - |
| **BLEU** | 생성 n-gram이 정답에 등장하는 비율 (정밀도) | 생성 n-gram 수 |
| **LLM-as-Judge** | LLM이 직접 관련성 채점 | - |

**내일 (260402)**: LLM-as-Judge를 활용한 다른 평가 메트릭들 계속